In [ ]:
# ============================================================
# DATEWISE: AI-POWERED SPEED DATING AND STRATEGIC MATCHMAKING
# Version 1.0 - Logistic Regression Baseline
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

print("\nLoading dataset...")

df = pd.read_csv(
    "Speed Dating Data.csv",
    encoding="cp1252"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 3. INITIAL DATA EXPLORATION
# ------------------------------------------------------------

print("\nFirst five rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nMatch distribution:")
print(df["match"].value_counts())

print("\nMatch percentages:")
print(df["match"].value_counts(normalize=True) * 100)


# ------------------------------------------------------------
# 4. SELECT RELEVANT PRE-DATE FEATURES
# ------------------------------------------------------------

# These are participants' interest ratings.
interest_columns = [
    "sports",
    "tvsports",
    "exercise",
    "dining",
    "museums",
    "art",
    "hiking",
    "gaming",
    "clubbing",
    "reading",
    "tv",
    "theater",
    "movies",
    "concerts",
    "music",
    "shopping",
    "yoga"
]

# Only keep columns required for our first model.
required_columns = [
    "iid",
    "pid",
    "wave",
    "age",
    "int_corr",
    "match"
] + interest_columns

df = df[required_columns].copy()


# ------------------------------------------------------------
# 5. REMOVE RECORDS WITHOUT PARTICIPANT IDENTIFIERS
# ------------------------------------------------------------

df = df.dropna(
    subset=["iid", "pid", "wave", "match"]
)

print("\nDataset shape after removing incomplete pair IDs:")
print(df.shape)

In [ ]:
# ------------------------------------------------------------
# 6. CREATE UNIQUE PAIR IDENTIFIERS
# ------------------------------------------------------------

# Each pair appears twice:
# A meeting B
# B meeting A
#
# We create a common identifier for both records.

df["person_1"] = np.minimum(df["iid"], df["pid"])
df["person_2"] = np.maximum(df["iid"], df["pid"])


# ------------------------------------------------------------
# 7. FEATURE ENGINEERING
# ------------------------------------------------------------

def create_pair_features(row_a, row_b):

    features = {}

    # Age difference between participants
    age_a = pd.to_numeric(row_a["age"], errors="coerce")
    age_b = pd.to_numeric(row_b["age"], errors="coerce")

    features["age_gap"] = abs(age_a - age_b)

    # Average of the pairwise interest correlation
    correlations = pd.to_numeric(
        pd.Series([row_a["int_corr"], row_b["int_corr"]]),
        errors="coerce"
    )

    features["int_corr"] = correlations.mean()

    # Compare participants' interests
    interest_differences = []

    for column in interest_columns:

        values = pd.to_numeric(
            pd.Series([row_a[column], row_b[column]]),
            errors="coerce"
        )

        if values.notna().all():

            difference = abs(values.iloc[0] - values.iloc[1])

        else:

            difference = np.nan

        features[column + "_difference"] = difference

        if pd.notna(difference):
            interest_differences.append(difference)

    # Overall interest similarity
    if interest_differences:

        average_difference = np.mean(interest_differences)

        features["interest_similarity"] = (
            1 - average_difference / 9
        )

    else:

        features["interest_similarity"] = np.nan

    return features


# ------------------------------------------------------------
# 8. CONVERT TWO-DIRECTIONAL RECORDS INTO ONE PAIR
# ------------------------------------------------------------

pair_records = []

group_columns = [
    "wave",
    "person_1",
    "person_2"
]

for _, group in df.groupby(group_columns):

    # We expect two reciprocal records for each pair.
    if len(group) != 2:
        continue

    row_a = group.iloc[0]
    row_b = group.iloc[1]

    # Both directions must have the same mutual-match target.
    if row_a["match"] != row_b["match"]:
        continue

    features = create_pair_features(row_a, row_b)

    features["match"] = int(row_a["match"])

    # Retain wave only for grouped train/test splitting.
    features["wave"] = row_a["wave"]

    pair_records.append(features)


pair_df = pd.DataFrame(pair_records)

print("\nPair-level dataset shape:")
print(pair_df.shape)

print("\nPair-level match distribution:")
print(pair_df["match"].value_counts())


# ------------------------------------------------------------
# 9. DEFINE INPUT FEATURES AND TARGET
# ------------------------------------------------------------

feature_columns = [
    "age_gap",
    "int_corr",
    "interest_similarity"
]

# Add individual interest differences.
feature_columns += [
    column + "_difference"
    for column in interest_columns
]

X = pair_df[feature_columns]

y = pair_df["match"]

# Wave is used only for splitting, never as a model feature.
groups = pair_df["wave"]
